<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/3_Live_Model_BiLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance deep-translator transformers tensorflow joblib beautifulsoup4

import yfinance as yf
import pandas as pd
import numpy as np
import joblib
import torch
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from deep_translator import GoogleTranslator
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tensorflow.keras.models import load_model

model = load_model('BBCA_BiLSTM_Sentiment_Model.keras')
scaler_features = joblib.load('scaler_features.pkl')
scaler_target = joblib.load('scaler_target.pkl')

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

In [ ]:
def get_lagged_sentiment(ticker="BBCA"):
    yesterday = (datetime.now() - timedelta(1)).strftime('%Y-%m-%d')
    rss_url = f"https://news.google.com/rss/search?q={ticker}+when:1d"

    response = requests.get(rss_url)
    soup = BeautifulSoup(response.content, 'xml')
    items = soup.find_all('item')

    headlines = [item.title.text for item in items]
    if not headlines:
        return 0.0 # Neutral if no news found

    translator = GoogleTranslator(source='id', target='en')
    translated_headlines = [translator.translate(h) for h in headlines[:10]] # Top 10

    inputs = tokenizer(translated_headlines, padding=True, truncation=True, return_tensors="pt")
    outputs = finbert(**inputs)
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

    # FinBERT labels: 0: Positive, 1: Negative, 2: Neutral
    # We convert to a single score: Positive=1, Neutral=0, Negative=-1
    scores = predictions.detach().numpy()
    sentiment_score = np.mean(scores[:, 0] - scores[:, 1])

    return sentiment_score

print("Sentiment Engine ready.")

In [ ]:
bbca = yf.Ticker("BBCA.JK")
today_data = bbca.history(period="1d")

if today_data.empty:
    print("Market is closed or data not available yet.")
else:
    iso_date = today_data.index[0].strftime('%Y-%m-%d')

    today_price = {
        'Date': iso_date,
        'Open': int(round(today_data['Open'].values[0])),
        'High': int(round(today_data['High'].values[0])),
        'Low': int(round(today_data['Low'].values[0])),
        'Close': int(round(today_data['Close'].values[0])),
        'Volume': int(today_data['Volume'].values[0])
    }

    sentiment = get_lagged_sentiment("BBCA")
    today_price['Sentiment_Score'] = sentiment

    master_df = pd.read_csv('BBCA_Master_Dataset_BiLSTM.csv')

    master_df['Date'] = pd.to_datetime(master_df['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
    master_df = master_df.dropna(subset=['Date']) # Clean up any broken rows just in case

    master_dates = master_df['Date'].astype(str).str.strip().values
    target_date = str(today_price['Date']).strip()

    new_row = pd.DataFrame([today_price])

    if target_date not in master_dates:
        master_df = pd.concat([master_df, new_row], ignore_index=True)
        master_df.to_csv('BBCA_Master_Dataset_BiLSTM.csv', index=False)
        print(f"Successfully added data for {target_date}")
    else:
        print(f"ℹData for {target_date} already exists. Skipping append.")

In [ ]:
import matplotlib.pyplot as plt

# 1. Prepare the 10-day window
recent_data = master_df.tail(10).copy()
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'Sentiment_Score']

# 2. Scale and Reshape
scaled_input = scaler_features.transform(recent_data[features])
scaled_input = scaled_input.reshape((1, 10, 6)) # (Batch, Window, Features)

# 3. Predict
predicted_scaled = model.predict(scaled_input, verbose=0)
predicted_price = scaler_target.inverse_transform(predicted_scaled)[0][0]

current_price = recent_data['Close'].values[-1]
current_date_str = recent_data['Date'].values[-1]

# Calculate Target Date (Next Business Day)
curr_dt = pd.to_datetime(current_date_str)
target_date_str = (curr_dt + pd.offsets.BDay()).strftime('%Y-%m-%d')

difference = predicted_price - current_price
verdict = "📈 UPTREND" if difference > 0 else "📉 DOWNTREND"

highest_sent_row = recent_data.loc[recent_data['Sentiment_Score'].idxmax()]
lowest_sent_row = recent_data.loc[recent_data['Sentiment_Score'].idxmin()]
avg_window_sentiment = recent_data['Sentiment_Score'].mean()

print("\n" + "="*50)
print(f" 📊    BBCA EXPLAINABLE AI DAILY DASHBOARD")
print("="*50)
print(f"Current Base Date : {current_date_str}")
print(f"Target Forecast   : {target_date_str} (Next Business Day)")
print(f"-"*50)
print(f"Current Price     : Rp {current_price:,.0f}")
print(f"Predicted Price   : Rp {predicted_price:,.0f}")
print(f"Expected Move     : Rp {difference:+,.0f}")
print(f"Market Verdict    : {verdict}")
print("="*50)
print(f"🧠 BI-LSTM DRIVING QUANTITATIVE INSIGHTS:")
print(f"• Window Avg Sentiment : {avg_window_sentiment:+.4f} (Range: -1 to +1)")
print(f"• Peak Optimism Date   : {highest_sent_row['Date']} (Score: {highest_sent_row['Sentiment_Score']:+.4f})")
print(f"• Peak Pessimism Date  : {lowest_sent_row['Date']} (Score: {lowest_sent_row['Sentiment_Score']:+.4f})")
print("="*50 + "\n")

fig, ax1 = plt.subplots(figsize=(12, 5))

color_price = '#1d3557'
ax1.set_xlabel('Historical Look-back Sequence (Days)')
ax1.set_ylabel('Close Price (IDR)', color=color_price)
ax1.plot(recent_data['Date'].values, recent_data['Close'].values, color=color_price, marker='o', linewidth=2.5, label='Actual Price')
ax1.tick_params(axis='y', labelcolor=color_price)
ax1.set_xticks(range(len(recent_data['Date'].values)))
ax1.set_xticklabels(recent_data['Date'].values, rotation=35, ha='right')

ax2 = ax1.twinx()
color_sent = '#2a9d8f' if avg_window_sentiment > 0 else '#e63946'
ax2.set_ylabel('Aggregated News Sentiment Score', color=color_sent)
ax2.bar(recent_data['Date'].values, recent_data['Sentiment_Score'].values, color=color_sent, alpha=0.3, label='Daily Sentiment')
ax2.tick_params(axis='y', labelcolor=color_sent)
ax2.set_ylim(-1.1, 1.1)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)

plt.title('Explainable AI: Price vs. News Sentiment Micro-Trends inside the Prediction Window', fontweight='bold', pad=15)
fig.tight_layout()
plt.show()